<a href="https://colab.research.google.com/github/msaiankitha/LearnSphere/blob/main/Live123.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost librosa imbalanced-learn gradio matplotlib sqlalchemy openai-whisper --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import random
import sqlite3
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import joblib
import gradio as gr
import whisper

from datetime import datetime
from collections import Counter
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier

In [ ]:
control_audio = "/content/drive/MyDrive/PittDataset/control/audio"
control_text = "/content/drive/MyDrive/PittDataset/control/transcript"

dementia_audio = "/content/drive/MyDrive/PittDataset/dementia/audio"
dementia_text = "/content/drive/MyDrive/PittDataset/dementia/transcript"

In [ ]:
print("Control audio exists:", os.path.exists(control_audio))
print("Control text exists:", os.path.exists(control_text))
print("Dementia audio exists:", os.path.exists(dementia_audio))
print("Dementia text exists:", os.path.exists(dementia_text))

if os.path.exists(control_audio):
    print("Control audio files:", len(os.listdir(control_audio)))
if os.path.exists(dementia_audio):
    print("Dementia audio files:", len(os.listdir(dementia_audio)))
if os.path.exists(control_text):
    print("Control transcript files:", len(os.listdir(control_text)))
if os.path.exists(dementia_text):
    print("Dementia transcript files:", len(os.listdir(dementia_text)))

Control audio exists: True
Control text exists: True
Dementia audio exists: True
Dementia text exists: True
Control audio files: 223
Dementia audio files: 307
Control transcript files: 243
Dementia transcript files: 309


In [ ]:
def clean_raw_text(raw_text):
    if raw_text is None:
        return ""

    text = raw_text.lower().strip()

    # normalize fillers
    text = re.sub(r"\b(um+|uh+|er+|ah+|hmm+|mmm+)\b", " um ", text)

    # keep only letters + spaces + simple punctuation
    text = re.sub(r"[^a-zA-Z\s'.!?]", " ", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# ============================================================
# ADD BELOW clean_raw_text(...)
# LIVE TEXT NORMALIZATION TO BETTER MATCH PITT-STYLE TEXT
# ============================================================

def normalize_live_text_for_model(text):
    """
    Make live typed text more similar to the style seen during training.
    This helps the text model generalize better.
    """
    if text is None:
        return ""

    # basic cleanup
    text = clean_raw_text(text)

    # collapse repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    # normalize repeated fillers
    text = re.sub(r"\b(um+|uh+|erm+|hmm+)\b", " um ", text, flags=re.IGNORECASE)

    # normalize repeated word sequences like:
    # "boy boy boy" -> "boy boy"
    words = text.split()
    reduced = []
    repeat_count = 0

    for i, w in enumerate(words):
        if i > 0 and w == words[i-1]:
            repeat_count += 1
            if repeat_count < 2:   # allow up to 2 repeats
                reduced.append(w)
        else:
            repeat_count = 0
            reduced.append(w)

    text = " ".join(reduced)

    return text

In [ ]:
def extract_patient_text_from_cha(file_path):
    patient_lines = []

    with open(file_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("*PAR:"):
                text = line.replace("*PAR:", "").strip()

                # remove CHAT annotations
                text = re.sub(r"\[[^\]]*\]", " ", text)
                text = re.sub(r"<[^>]*>", " ", text)
                text = re.sub(r"&=\w+", " ", text)
                text = re.sub(r"\([^)]*\)", " ", text)

                # keep readable content
                text = re.sub(r"[^a-zA-Z\s'.!?]", " ", text)
                text = re.sub(r"\s+", " ", text).strip().lower()

                if text:
                    patient_lines.append(text)

    return " ".join(patient_lines)

In [ ]:
def extract_linguistic_features(text):
    text = clean_raw_text(text)

    words = text.split()
    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    word_count = len(words)
    unique_words = len(set(words))
    ttr = unique_words / word_count if word_count > 0 else 0

    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    avg_sentence_len = word_count / len(sentences) if len(sentences) > 0 else 0

    filler_words = {"um", "uh", "er", "ah", "hmm", "mmm"}
    filler_count = sum(1 for w in words if w in filler_words)

    pronouns = {"he", "she", "they", "it", "him", "her", "them"}
    pronoun_count = sum(1 for w in words if w in pronouns)

    repetition_score = 0
    if len(words) > 1:
        repetition_score = sum(1 for i in range(1, len(words)) if words[i] == words[i-1])

    vocab_richness = unique_words / np.sqrt(word_count) if word_count > 0 else 0

    common_ratio = 0
    if word_count > 0:
        c = Counter(words)
        top3 = sum(freq for _, freq in c.most_common(3))
        common_ratio = top3 / word_count

    feats = np.array([
        word_count,
        unique_words,
        ttr,
        avg_word_len,
        avg_sentence_len,
        filler_count,
        pronoun_count,
        repetition_score,
        vocab_richness,
        common_ratio
    ], dtype=np.float32)

    return feats

In [ ]:
def make_asr_style_text(text):
    text = clean_raw_text(text)

    # remove punctuation
    text = re.sub(r"[.!?]", " ", text)
    words = text.split()

    drop_words = {"the", "a", "an", "is", "are", "was", "were"}
    new_words = []

    for w in words:
        if w in drop_words and random.random() < 0.2:
            continue
        new_words.append(w)

    # occasional filler insertion
    if len(new_words) > 5 and random.random() < 0.3:
        idx = random.randint(0, len(new_words)-1)
        new_words.insert(idx, "um")

    return " ".join(new_words)

In [ ]:
def augment_audio_signal(y, sr):
    y_aug = y.copy()

    # random volume
    gain = np.random.uniform(0.8, 1.2)
    y_aug = y_aug * gain

    # add mild noise
    noise_amp = 0.002 * np.random.uniform(0.5, 1.5)
    noise = np.random.randn(len(y_aug)) * noise_amp
    y_aug = y_aug + noise

    # slight speed change
    speed_factor = np.random.uniform(0.95, 1.05)
    try:
        y_aug = librosa.effects.time_stretch(y_aug, rate=speed_factor)
    except:
        pass

    # pad to minimum 3 sec
    if len(y_aug) < sr * 3:
        y_aug = np.pad(y_aug, (0, sr * 3 - len(y_aug)), mode='constant')

    # normalize
    max_amp = np.max(np.abs(y_aug))
    if max_amp > 0:
        y_aug = y_aug / max_amp

    return y_aug

In [ ]:
def extract_audio_features_from_signal(y, sr=16000):
    if len(y) == 0:
        raise ValueError("Empty audio signal.")

    y = y - np.mean(y)

    max_amp = np.max(np.abs(y))
    if max_amp > 0:
        y = y / max_amp

    y, _ = librosa.effects.trim(y, top_db=20)

    min_len = sr * 3
    if len(y) < min_len:
        y = np.pad(y, (0, min_len - len(y)), mode='constant')

    frame_length = int(0.025 * sr)
    hop_length = int(0.010 * sr)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    delta_mfcc = librosa.feature.delta(mfcc)
    delta_mfcc_mean = np.mean(delta_mfcc, axis=1)
    delta_mfcc_std = np.std(delta_mfcc, axis=1)

    # Spectral
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]

    spec_centroid_mean = np.mean(spectral_centroid)
    spec_centroid_std = np.std(spectral_centroid)

    spec_bandwidth_mean = np.mean(spectral_bandwidth)
    spec_bandwidth_std = np.std(spectral_bandwidth)

    spec_rolloff_mean = np.mean(spectral_rolloff)
    spec_rolloff_std = np.std(spectral_rolloff)

    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)

    rms_mean = np.mean(rms)
    rms_std = np.std(rms)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    chroma_std = np.std(chroma, axis=1)

    # Spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    spectral_contrast_mean = np.mean(spectral_contrast, axis=1)
    spectral_contrast_std = np.std(spectral_contrast, axis=1)

    # Pitch
    try:
        pitch = librosa.yin(y, fmin=50, fmax=300, sr=sr, frame_length=frame_length)
        pitch = pitch[~np.isnan(pitch)]
        pitch = pitch[pitch > 0]

        pitch_mean = np.mean(pitch) if len(pitch) > 0 else 0
        pitch_std = np.std(pitch) if len(pitch) > 0 else 0
        pitch_range = (np.max(pitch) - np.min(pitch)) if len(pitch) > 0 else 0

        pitch_diff = np.abs(np.diff(pitch)) if len(pitch) > 1 else np.array([0.0])
        jitter_proxy = np.mean(pitch_diff) if len(pitch_diff) > 0 else 0
    except:
        pitch_mean = 0
        pitch_std = 0
        pitch_range = 0
        jitter_proxy = 0

    # Tempo
    try:
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        if isinstance(tempo, np.ndarray):
            tempo = float(np.mean(tempo))
        else:
            tempo = float(tempo)
    except:
        tempo = 0.0

    duration = librosa.get_duration(y=y, sr=sr)

    if len(rms) == 0:
        rms = np.array([0.0])

    threshold = max(0.02, np.percentile(rms, 25) * 0.8)
    voiced_mask = rms > threshold

    voiced_ratio = np.mean(voiced_mask)
    silence_ratio = 1.0 - voiced_ratio

    # segments
    speech_segments = 0
    pause_segments = 0
    in_speech = False
    in_pause = False

    for v in voiced_mask:
        if v and not in_speech:
            in_speech = True
            speech_segments += 1
        elif not v:
            in_speech = False

        if not v and not in_pause:
            in_pause = True
            pause_segments += 1
        elif v:
            in_pause = False

    # pause durations
    pause_lengths = []
    current_pause = 0
    for v in voiced_mask:
        if not v:
            current_pause += 1
        else:
            if current_pause > 0:
                pause_lengths.append(current_pause)
                current_pause = 0
    if current_pause > 0:
        pause_lengths.append(current_pause)

    pause_durations = [p * hop_length / sr for p in pause_lengths]
    avg_pause_duration = np.mean(pause_durations) if pause_durations else 0
    max_pause_duration = np.max(pause_durations) if pause_durations else 0
    total_pause_time = np.sum(pause_durations) if pause_durations else 0

    # speech durations
    speech_lengths = []
    current_speech = 0
    for v in voiced_mask:
        if v:
            current_speech += 1
        else:
            if current_speech > 0:
                speech_lengths.append(current_speech)
                current_speech = 0
    if current_speech > 0:
        speech_lengths.append(current_speech)

    speech_durations = [s * hop_length / sr for s in speech_lengths]
    avg_speech_segment_duration = np.mean(speech_durations) if speech_durations else 0

    speaking_rate_proxy = np.sum(voiced_mask) / duration if duration > 0 else 0
    pause_rate = pause_segments / duration if duration > 0 else 0
    speech_segment_rate = speech_segments / duration if duration > 0 else 0

    rms_diff = np.abs(np.diff(rms)) if len(rms) > 1 else np.array([0.0])
    shimmer_proxy = np.mean(rms_diff) if len(rms_diff) > 0 else 0

    features = np.hstack([
        mfcc_mean, mfcc_std, delta_mfcc_mean, delta_mfcc_std,
        chroma_mean, chroma_std,
        spectral_contrast_mean, spectral_contrast_std,
        [spec_centroid_mean], [spec_centroid_std],
        [spec_bandwidth_mean], [spec_bandwidth_std],
        [spec_rolloff_mean], [spec_rolloff_std],
        [zcr_mean], [zcr_std],
        [rms_mean], [rms_std],
        [pitch_mean], [pitch_std], [pitch_range],
        [jitter_proxy], [shimmer_proxy],
        [tempo], [duration],
        [voiced_ratio], [silence_ratio],
        [speech_segments], [pause_segments],
        [avg_pause_duration], [max_pause_duration], [total_pause_time],
        [avg_speech_segment_duration], [speaking_rate_proxy],
        [pause_rate], [speech_segment_rate]
    ])

    return np.array(features, dtype=np.float32)

In [ ]:
def extract_audio_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    return extract_audio_features_from_signal(y, sr)

In [ ]:
# ============================================================
# ADD BELOW extract_audio_features(...)
# LIVE AUDIO PREPROCESSING FOR STRONGER REAL-WORLD INFERENCE
# ============================================================

def preprocess_live_audio(audio_file, target_sr=16000):
    y, sr = librosa.load(audio_file, sr=target_sr)

    if len(y) == 0:
        raise ValueError("Empty audio file.")

    # Remove DC offset
    y = y - np.mean(y)

    # Trim leading and trailing silence
    y, _ = librosa.effects.trim(y, top_db=25)

    # Minimum 5 seconds after trimming
    min_len = target_sr * 5
    if len(y) < min_len:
        y = np.pad(y, (0, min_len - len(y)), mode="constant")

    # Normalize amplitude
    max_amp = np.max(np.abs(y))
    if max_amp > 0:
        y = y / max_amp

    return y, target_sr

In [ ]:
# ============================================================
# ADD BELOW preprocess_live_audio(...)
# LIVE AUDIO DOMAIN-SHIFT COMPENSATION
# ============================================================

def calibrate_live_audio_score(score):
    """
    Mild upward correction because Pitt-trained audio models
    often under-score real live microphone recordings.
    """
    adjusted = 0.10 + (0.90 * score)
    return min(1.0, max(0.0, adjusted))

In [ ]:
sample_control_files = [f for f in os.listdir(control_audio) if f.lower().endswith(".wav")]
sample_file = os.path.join(control_audio, sample_control_files[0])

sample_feat = extract_audio_features(sample_file)
print("Sample file:", sample_file)
print("Feature length:", len(sample_feat))

Sample file: /content/drive/MyDrive/PittDataset/control/audio/002-0.wav
Feature length: 146


In [ ]:
EXPECTED_AUDIO_FEATURES = 146

audio_features = []
audio_labels = []

def add_audio_file(path, label, add_augmented=True):
    try:
        # original
        feat = extract_audio_features(path)
        if len(feat) == EXPECTED_AUDIO_FEATURES:
            audio_features.append(feat)
            audio_labels.append(label)

        # augmented copies
        if add_augmented:
            y, sr = librosa.load(path, sr=16000)

            for _ in range(2):  # 2 augmented copies per file
                y_aug = augment_audio_signal(y, sr)
                feat_aug = extract_audio_features_from_signal(y_aug, sr)

                if len(feat_aug) == EXPECTED_AUDIO_FEATURES:
                    audio_features.append(feat_aug)
                    audio_labels.append(label)

    except Exception as e:
        print("Skipping:", path, "|", e)

# Control
for file in os.listdir(control_audio):
    if file.lower().endswith(".wav"):
        add_audio_file(os.path.join(control_audio, file), label=0, add_augmented=True)

# Dementia
for file in os.listdir(dementia_audio):
    if file.lower().endswith(".wav"):
        add_audio_file(os.path.join(dementia_audio, file), label=1, add_augmented=True)

X_audio = np.array(audio_features, dtype=np.float32)
y_audio = np.array(audio_labels)

print("Audio dataset shape:", X_audio.shape)
print("Control:", np.sum(y_audio == 0))
print("Dementia:", np.sum(y_audio == 1))

Audio dataset shape: (1590, 146)
Control: 669
Dementia: 921


In [ ]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_audio, y_audio,
    test_size=0.2,
    random_state=42,
    stratify=y_audio
)

audio_scaler = StandardScaler()
X_train_a = audio_scaler.fit_transform(X_train_a)
X_test_a = audio_scaler.transform(X_test_a)

print("Audio scaler expects:", audio_scaler.n_features_in_)

# class weights (NO SMOTE)
classes = np.unique(y_train_a)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_a)
weight_dict = {cls: wt for cls, wt in zip(classes, class_weights)}
sample_weights = np.array([weight_dict[y] for y in y_train_a])

audio_model = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.75,
    min_child_weight=4,
    gamma=0.4,
    reg_lambda=4.0,
    reg_alpha=1.5,
    random_state=42,
    eval_metric='logloss'
)

audio_model.fit(X_train_a, y_train_a, sample_weight=sample_weights)

# probabilities
audio_probs = audio_model.predict_proba(X_test_a)[:, 1]

Audio scaler expects: 146


In [ ]:
best_thresh = 0.5
best_bal = 0

for thresh in np.arange(0.45, 0.76, 0.05):
    pred_thresh = (audio_probs >= thresh).astype(int)
    bal = balanced_accuracy_score(y_test_a, pred_thresh)

    print(f"\nThreshold = {thresh:.2f}")
    print("Accuracy:", round(accuracy_score(y_test_a, pred_thresh), 4))
    print("Balanced Accuracy:", round(bal, 4))
    print(confusion_matrix(y_test_a, pred_thresh))

    if bal > best_bal:
        best_bal = bal
        best_thresh = float(thresh)

print("\nBest Audio Threshold:", best_thresh)
print("Best Audio Balanced Accuracy:", round(best_bal, 4))

In [ ]:
AUDIO_THRESHOLD = best_thresh

audio_pred = (audio_probs >= AUDIO_THRESHOLD).astype(int)

print("Audio Accuracy:", accuracy_score(y_test_a, audio_pred))
print("Audio Balanced Accuracy:", balanced_accuracy_score(y_test_a, audio_pred))
print("\nAudio Confusion Matrix:\n", confusion_matrix(y_test_a, audio_pred))
print("\nAudio Classification Report:\n")
print(classification_report(y_test_a, audio_pred))

Audio Accuracy: 0.7169811320754716
Audio Balanced Accuracy: 0.7341417910447761

Audio Confusion Matrix:
 [[113  21]
 [ 69 115]]

Audio Classification Report:

              precision    recall  f1-score   support

           0       0.62      0.84      0.72       134
           1       0.85      0.62      0.72       184

    accuracy                           0.72       318
   macro avg       0.73      0.73      0.72       318
weighted avg       0.75      0.72      0.72       318



In [ ]:
joblib.dump(audio_model, "audio_model.pkl")
joblib.dump(audio_scaler, "audio_scaler.pkl")
joblib.dump(AUDIO_THRESHOLD, "audio_threshold.pkl")

print("Saved audio_model.pkl, audio_scaler.pkl, audio_threshold.pkl")

Saved audio_model.pkl, audio_scaler.pkl, audio_threshold.pkl


In [ ]:
text_docs = []
text_labels = []

# Control transcripts
for file in os.listdir(control_text):
    if file.endswith(".cha"):
        path = os.path.join(control_text, file)
        try:
            txt = extract_patient_text_from_cha(path)
            if len(txt.strip()) > 0:
                text_docs.append(txt)
                text_labels.append(0)
        except Exception as e:
            print("Skipping control text:", file, "|", e)

# Dementia transcripts
for file in os.listdir(dementia_text):
    if file.endswith(".cha"):
        path = os.path.join(dementia_text, file)
        try:
            txt = extract_patient_text_from_cha(path)
            if len(txt.strip()) > 0:
                text_docs.append(txt)
                text_labels.append(1)
        except Exception as e:
            print("Skipping dementia text:", file, "|", e)

print("Original text docs:", len(text_docs))
print("Control:", sum(1 for x in text_labels if x == 0))
print("Dementia:", sum(1 for x in text_labels if x == 1))

Original text docs: 552
Control: 243
Dementia: 309


In [ ]:
aug_text_docs = []
aug_text_labels = []

for txt, label in zip(text_docs, text_labels):
    clean_txt = clean_raw_text(txt)

    # original clean text
    aug_text_docs.append(clean_txt)
    aug_text_labels.append(label)

    # ASR-style version
    aug_text_docs.append(make_asr_style_text(clean_txt))
    aug_text_labels.append(label)

print("Expanded text docs:", len(aug_text_docs))

In [ ]:
# Word TF-IDF
word_tfidf = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1, 3),
    stop_words="english",
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

# Character TF-IDF
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=5000,
    min_df=1,
    sublinear_tf=True
)

X_word = word_tfidf.fit_transform(aug_text_docs)
X_char = char_tfidf.fit_transform(aug_text_docs)
X_ling = np.array([extract_linguistic_features(t) for t in aug_text_docs], dtype=np.float32)

X_text_final = hstack([
    X_word,
    X_char,
    csr_matrix(X_ling)
])

y_text = np.array(aug_text_labels)

print("Word TF-IDF shape:", X_word.shape)
print("Char TF-IDF shape:", X_char.shape)
print("Linguistic shape:", X_ling.shape)
print("Final text shape:", X_text_final.shape)

In [ ]:
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text_final,
    y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)

text_model = LogisticRegression(
    C=3.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

text_model.fit(X_train_t, y_train_t)

text_pred = text_model.predict(X_test_t)

print("Text Accuracy:", accuracy_score(y_test_t, text_pred))
print("Text Balanced Accuracy:", balanced_accuracy_score(y_test_t, text_pred))
print("\nText Confusion Matrix:\n", confusion_matrix(y_test_t, text_pred))
print("\nText Classification Report:\n")
print(classification_report(y_test_t, text_pred))

In [ ]:
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")
print("Whisper loaded.")

In [ ]:
# ============================================================
# OPTIONAL REPLACE: BETTER WHISPER FOR DISPLAY ONLY
# (NOT USED FOR FINAL SCORING)
# ============================================================

def transcribe_audio(audio_file):
    if audio_file is None:
        return ""

    try:
        result = whisper_model.transcribe(
            audio_file,
            fp16=False,
            language="en"
        )
        return result.get("text", "").strip()
    except Exception as e:
        print("Whisper transcription error:", e)
        return ""

In [ ]:
def interpret_risk(score):
    if score >= 0.62:
        return "High Risk"
    elif score >= 0.4:
        return "Moderate Risk"
    else:
        return "Low Risk"

In [ ]:
# ============================================================
# REPLACE OLD predict_audio_only() COMPLETELY
# ============================================================

def predict_audio_only(audio_file):
    if audio_file is None:
        raise ValueError("Please provide an audio file.")

    # Strong live-audio preprocessing
    y, sr = preprocess_live_audio(audio_file, target_sr=16000)

    # Duration after trimming
    duration = librosa.get_duration(y=y, sr=sr)
    if duration < 5:
        raise ValueError("Please speak for at least 5–8 seconds for reliable audio analysis.")

    # Extract features directly from cleaned signal
    feat = extract_audio_features_from_signal(y, sr)

    # Safety check
    if len(feat) != audio_scaler.n_features_in_:
        raise ValueError(
            f"Audio feature mismatch: got {len(feat)}, expected {audio_scaler.n_features_in_}"
        )

    # Scale
    feat_scaled = audio_scaler.transform([feat])

    # Raw model score
    raw_score = float(audio_model.predict_proba(feat_scaled)[0][1])

    # Live calibration
    score = calibrate_live_audio_score(raw_score)

    label = 1 if score >= AUDIO_THRESHOLD else 0
    return score, label

In [ ]:
# ============================================================
# REPLACE OLD predict_text_only() COMPLETELY
# LIVE TEXT STRONGER / MORE ROBUST
# ============================================================

def predict_text_only(raw_text):
    if raw_text is None or raw_text.strip() == "":
        raise ValueError("Please provide text.")

    # Normalize live text to better match training style
    text = normalize_live_text_for_model(raw_text)

    # Build same text features as training
    Xw = word_tfidf.transform([text])
    Xc = char_tfidf.transform([text])

    ling = np.array([extract_linguistic_features(text)], dtype=np.float32)
    ling_sparse = csr_matrix(ling)

    X = hstack([Xw, Xc, ling_sparse])

    score = float(text_model.predict_proba(X)[0][1])

    # Small live-text calibration (much smaller than audio)
    # because live typed text can still be clean but slightly out-of-domain
    score = min(1.0, max(0.0, 0.05 + 0.95 * score))

    label = 1 if score >= 0.50 else 0
    return score, label

In [ ]:
# ============================================================
# REPLACE OLD get_text_confidence() COMPLETELY
# IMPORTANT: THIS VERSION TAKES 2 ARGUMENTS
# ============================================================

def get_text_confidence(raw_text, text_score):
    """
    Confidence of text branch using:
    - score certainty (distance from 0.5)
    - text length / richness
    """

    if raw_text is None:
        raw_text = ""

    text = normalize_live_text_for_model(raw_text)
    words = text.split()
    n_words = len(words)

    # lexical richness
    unique_ratio = len(set(words)) / max(1, n_words)

    # certainty from score
    score_certainty = 0.5 + abs(text_score - 0.5)

    # length factor (saturates around 25 words)
    length_factor = min(1.0, n_words / 25.0)

    # combine
    conf = 0.6 * score_certainty + 0.25 * length_factor + 0.15 * unique_ratio

    return float(min(1.0, max(0.1, conf)))

In [ ]:
# ============================================================
# REPLACE OLD get_audio_confidence() COMPLETELY
# ============================================================

def get_audio_confidence(audio_score):
    """
    Confidence of audio branch based on how far score is from uncertainty (0.5).
    """
    return 0.5 + abs(audio_score - 0.5)

In [ ]:
# ============================================================
# REPLACE OLD predict_by_mode() COMPLETELY
# NO MORE AUTO-TEXT FUSION FROM WHISPER
# ============================================================

def predict_by_mode(mode, audio_file=None, raw_text=None):
    """
    Modes:
    - Audio Only   -> uses only audio
    - Text Only    -> uses only typed text
    - Audio + Text -> requires BOTH audio and typed text
    """

    mode = mode.strip()

    if mode == "Audio Only":
        if audio_file is None:
            raise ValueError("Please provide audio for Audio Only mode.")

        audio_score, audio_label = predict_audio_only(audio_file)

        return {
            "final_score": audio_score,
            "risk_label": interpret_risk(audio_score),
            "mode": "Audio Only",
            "audio_score": audio_score,
            "text_score": None,
            "used_text": ""
        }

    elif mode == "Text Only":
        if raw_text is None or raw_text.strip() == "":
            raise ValueError("Please provide text for Text Only mode.")

        text_score, text_label = predict_text_only(raw_text)

        return {
            "final_score": text_score,
            "risk_label": interpret_risk(text_score),
            "mode": "Text Only",
            "audio_score": None,
            "text_score": text_score,
            "used_text": raw_text
        }

    elif mode == "Audio + Text":
        if audio_file is None:
            raise ValueError("Please provide audio for Audio + Text mode.")

        if raw_text is None or raw_text.strip() == "":
            raise ValueError(
                "Audio + Text mode requires BOTH audio and typed text.\n"
                "If you only want live speech, use Audio Only mode."
            )

        # Audio branch
        audio_score, audio_label = predict_audio_only(audio_file)

        # Text branch (ONLY typed text, NO auto Whisper scoring)
        text_score, text_label = predict_text_only(raw_text)

        # Adaptive fusion
        audio_conf = get_audio_confidence(audio_score)
        text_conf = get_text_confidence(raw_text, text_score)

        total_conf = audio_conf + text_conf + 1e-8
        wa = audio_conf / total_conf
        wt = text_conf / total_conf

        final_score = wa * audio_score + wt * text_score

        return {
            "final_score": final_score,
            "risk_label": interpret_risk(final_score),
            "mode": f"Audio + Text (Adaptive Fusion | wa={wa:.2f}, wt={wt:.2f})",
            "audio_score": audio_score,
            "text_score": text_score,
            "used_text": raw_text
        }

    else:
        raise ValueError("Invalid mode. Choose Audio Only, Text Only, or Audio + Text.")

In [ ]:
# ============================================================
# NEW DATABASE SCHEMA (AUTH + DASHBOARD)
# ============================================================

import hashlib

DB_PATH = "cognitive_risk_dashboard.db"

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    # Users table (doctor + patient)
    c.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        role TEXT NOT NULL,              -- 'doctor' or 'patient'
        full_name TEXT NOT NULL,
        username TEXT UNIQUE,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        created_at TEXT NOT NULL
    )
    """)

    # Doctor-patient mapping
    c.execute("""
    CREATE TABLE IF NOT EXISTS doctor_patients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        doctor_email TEXT NOT NULL,
        patient_email TEXT NOT NULL
    )
    """)

    # Assessments table
    c.execute("""
    CREATE TABLE IF NOT EXISTS assessments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_email TEXT NOT NULL,
        timestamp TEXT NOT NULL,
        mode TEXT NOT NULL,
        audio_score REAL,
        text_score REAL,
        final_score REAL NOT NULL,
        risk_label TEXT NOT NULL,
        used_text TEXT,
        alert_level TEXT
    )
    """)

    conn.commit()
    conn.close()

init_db()
print("✅ New dashboard DB initialized.")

✅ New dashboard DB initialized.


In [ ]:
# ============================================================
# AUTH FUNCTIONS
# ============================================================

def register_user(role, full_name, username, email, password):
    if not role or role not in ["doctor", "patient"]:
        return "❌ Invalid role."

    if not full_name.strip():
        return "❌ Full name required."

    if not email.strip():
        return "❌ Email required."

    if not password.strip():
        return "❌ Password required."

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    try:
        c.execute("""
            INSERT INTO users (role, full_name, username, email, password_hash, created_at)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            role,
            full_name.strip(),
            username.strip() if username else None,
            email.strip().lower(),
            hash_password(password.strip()),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ))
        conn.commit()
        msg = f"✅ {role.title()} registered successfully."
    except sqlite3.IntegrityError:
        msg = "❌ Email or username already exists."
    finally:
        conn.close()

    return msg


def login_user(login_id, password, role):
    """
    login_id can be email or username
    """
    if not login_id.strip() or not password.strip():
        return None, "❌ Please enter login and password."

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute("""
        SELECT id, role, full_name, username, email
        FROM users
        WHERE role = ?
          AND password_hash = ?
          AND (LOWER(email) = LOWER(?) OR LOWER(username) = LOWER(?))
    """, (
        role,
        hash_password(password.strip()),
        login_id.strip().lower(),
        login_id.strip().lower()
    ))

    row = c.fetchone()
    conn.close()

    if row:
        user = {
            "id": row[0],
            "role": row[1],
            "full_name": row[2],
            "username": row[3],
            "email": row[4]
        }
        return user, f"✅ Login successful. Welcome {user['full_name']}!"
    else:
        return None, "❌ Invalid credentials."

In [ ]:
# ============================================================
# REPLACE OLD link_patient_to_doctor(...) WITH THIS
# ============================================================

def link_patient_to_doctor_flexible(doctor_id, patient_id):
    """
    Allows patient to continue even if already linked.
    Returns:
      - status message
      - whether assessment section should be shown
    """

    if not doctor_id or not patient_id:
        return "❌ Please enter both doctor and patient email/username.", gr.update(visible=False)

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # Find doctor by email or username
    cur.execute("""
        SELECT id, email, username FROM users
        WHERE role='doctor' AND (email=? OR username=?)
    """, (doctor_id.strip(), doctor_id.strip()))
    doctor = cur.fetchone()

    # Find patient by email or username
    cur.execute("""
        SELECT id, email, username FROM users
        WHERE role='patient' AND (email=? OR username=?)
    """, (patient_id.strip(), patient_id.strip()))
    patient = cur.fetchone()

    if not doctor:
        conn.close()
        return "❌ Doctor not found.", gr.update(visible=False)

    if not patient:
        conn.close()
        return "❌ Patient not found.", gr.update(visible=False)

    doctor_db_id = doctor[0]
    patient_db_id = patient[0]

    # Check existing link
    cur.execute("""
        SELECT id FROM doctor_patient_links
        WHERE doctor_id=? AND patient_id=?
    """, (doctor_db_id, patient_db_id))
    existing = cur.fetchone()

    if existing:
        conn.close()
        # IMPORTANT: still allow assessment
        return "⚠️ Patient already linked to doctor. You can continue to assessment.", gr.update(visible=True)

    # Create new link
    cur.execute("""
        INSERT INTO doctor_patient_links (doctor_id, patient_id)
        VALUES (?, ?)
    """, (doctor_db_id, patient_db_id))
    conn.commit()
    conn.close()

    return "✅ Linked successfully. You can now continue to assessment.", gr.update(visible=True)

In [ ]:
# ============================================================
# ALERT LOGIC + SAVE ASSESSMENT
# ============================================================

def compute_alert_level(score):
    if score >= 0.78:
        return "CRITICAL"
    elif score >= 0.6:
        return "HIGH"
    elif score >= 0.4:
        return "MODERATE"
    else:
        return "LOW"


def save_assessment(patient_email, result):
    if not patient_email or patient_email.strip() == "":
        raise ValueError("Patient email is required to save assessment.")

    patient_email = patient_email.strip().lower()
    alert_level = compute_alert_level(result["final_score"])

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute("""
        INSERT INTO assessments (
            patient_email, timestamp, mode, audio_score, text_score,
            final_score, risk_label, used_text, alert_level
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        patient_email,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        result["mode"],
        result["audio_score"],
        result["text_score"],
        result["final_score"],
        result["risk_label"],
        result["used_text"],
        alert_level
    ))

    conn.commit()
    conn.close()

In [ ]:
# ============================================================
# REPLACE OLD patient_analyze() COMPLETELY
# SHOWS TRANSCRIPT FOR AUDIO-ONLY, BUT DOESN'T SCORE WITH IT
# ============================================================

def patient_analyze(mode, patient_email, audio_file, text_input):
    try:
        result = predict_by_mode(mode=mode, audio_file=audio_file, raw_text=text_input)

        # For display only
        display_text = result["used_text"]

        # In Audio Only mode, show transcript but do NOT use it for scoring
        if mode == "Audio Only" and audio_file is not None:
            display_text = transcribe_audio(audio_file)

        # Save to DB
        save_result = {
            "final_score": result["final_score"],
            "risk_label": result["risk_label"],
            "mode": result["mode"],
            "audio_score": result["audio_score"],
            "text_score": result["text_score"],
            "used_text": display_text
        }

        save_assessment(patient_email, save_result)

        return (
            f"{result['final_score']:.4f}",
            result["risk_label"],
            result["mode"],
            "" if result["audio_score"] is None else f"{result['audio_score']:.4f}",
            "" if result["text_score"] is None else f"{result['text_score']:.4f}",
            display_text,
            f"🚨 Alert Level: {compute_alert_level(result['final_score'])}"
        )

    except Exception as e:
        return ("0.0000", f"Error: {str(e)}", "Error", "", "", "", "❌ No alert")

In [ ]:
# ============================================================
# DASHBOARD DATA FUNCTIONS
# ============================================================

def get_doctor_patients(doctor_email):
    doctor_email = doctor_email.strip().lower()

    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT u.full_name, u.email
        FROM doctor_patients dp
        JOIN users u ON dp.patient_email = u.email
        WHERE dp.doctor_email = ?
        ORDER BY u.full_name
    """
    df = pd.read_sql_query(query, conn, params=(doctor_email,))
    conn.close()

    if df.empty:
        return pd.DataFrame({"Message": ["No linked patients found."]})
    return df


def get_patient_assessments(patient_email):
    patient_email = patient_email.strip().lower()

    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT timestamp, mode, audio_score, text_score, final_score, risk_label, alert_level
        FROM assessments
        WHERE patient_email = ?
        ORDER BY timestamp ASC
    """, conn, params=(patient_email,))
    conn.close()

    return df


def get_latest_patient_summary(patient_email):
    df = get_patient_assessments(patient_email)

    if df.empty:
        return {
            "latest_score": "N/A",
            "risk_label": "No data",
            "alert_level": "No data",
            "trend": "No history"
        }

    latest = df.iloc[-1]

    trend = "Stable"
    if len(df) >= 2:
        prev = df.iloc[-2]["final_score"]
        curr = latest["final_score"]
        if curr - prev >= 0.10:
            trend = "Worsening"
        elif prev - curr >= 0.10:
            trend = "Improving"

    return {
        "latest_score": f"{latest['final_score']:.4f}",
        "risk_label": latest["risk_label"],
        "alert_level": latest["alert_level"],
        "trend": trend
    }

In [ ]:
# ============================================================
# VISUALS FOR DASHBOARD
# ============================================================

def memory_decline_chart(patient_email):
    df = get_patient_assessments(patient_email)

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor("#eaf7ff")
    ax.set_facecolor("#f8fdff")

    if df.empty:
        ax.text(0.5, 0.5, "No assessment history found", ha="center", va="center", fontsize=14)
        ax.axis("off")
        return fig

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    ax.plot(df["timestamp"], df["final_score"], marker="o", linewidth=2)

    ax.set_title("Memory Decline / Risk Tracking Over Time", fontsize=14, fontweight="bold")
    ax.set_xlabel("Assessment Time")
    ax.set_ylabel("Risk Score")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    # Alert thresholds
    ax.axhline(0.5, linestyle="--", alpha=0.5)
    ax.axhline(0.7, linestyle="--", alpha=0.5)
    ax.axhline(0.85, linestyle="--", alpha=0.5)

    plt.xticks(rotation=30)
    plt.tight_layout()
    return fig


def risk_gauge(score):
    try:
        score = float(score)
    except:
        score = 0.0

    fig, ax = plt.subplots(figsize=(7, 4), subplot_kw={'projection': 'polar'})
    fig.patch.set_facecolor("#eaf7ff")
    ax.set_facecolor("#f8fdff")

    ax.set_theta_offset(np.pi)
    ax.set_theta_direction(-1)

    # low
    theta = np.linspace(0, np.pi * 0.50, 100)
    ax.plot(theta, np.ones_like(theta), linewidth=18)

    # moderate
    theta = np.linspace(np.pi * 0.50, np.pi * 0.75, 100)
    ax.plot(theta, np.ones_like(theta), linewidth=18)

    # high
    theta = np.linspace(np.pi * 0.75, np.pi, 100)
    ax.plot(theta, np.ones_like(theta), linewidth=18)

    theta_score = np.pi * score
    ax.arrow(theta_score, 0, 0, 0.8, width=0.03)

    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_title(f"Risk Gauge\nScore: {score:.2f}", va='bottom')
    return fig

In [ ]:
# ============================================================
# DOCTOR VIEW HELPERS
# ============================================================

def doctor_patient_dropdown_choices(doctor_email):
    df = get_doctor_patients(doctor_email)

    if "Message" in df.columns:
        return gr.update(choices=[], value=None)

    choices = [f"{row['full_name']} | {row['email']}" for _, row in df.iterrows()]
    return gr.update(choices=choices, value=choices[0] if choices else None)


def doctor_view_patient(selected_patient):
    if not selected_patient:
        empty_df = pd.DataFrame({"Message": ["No patient selected"]})
        return (
            "N/A", "No data", "No data", "No history",
            empty_df, None, None, "No alerts"
        )

    patient_email = selected_patient.split("|")[-1].strip()
    summary = get_latest_patient_summary(patient_email)
    df = get_patient_assessments(patient_email)

    alert_msg = "No alerts"
    if not df.empty:
        latest_alert = df.iloc[-1]["alert_level"]
        if latest_alert == "CRITICAL":
            alert_msg = "🚨 CRITICAL: Immediate doctor attention required."
        elif latest_alert == "HIGH":
            alert_msg = "⚠️ HIGH: Patient condition needs close monitoring."
        elif latest_alert == "MODERATE":
            alert_msg = "🟡 MODERATE: Continue follow-up."

    gauge_fig = None
    if summary["latest_score"] != "N/A":
        gauge_fig = risk_gauge(float(summary["latest_score"]))

    trend_fig = memory_decline_chart(patient_email)

    return (
        summary["latest_score"],
        summary["risk_label"],
        summary["alert_level"],
        summary["trend"],
        df if not df.empty else pd.DataFrame({"Message": ["No assessments found"]}),
        gauge_fig,
        trend_fig,
        alert_msg
    )

In [ ]:
# ============================================================
# ADD THIS CELL BEFORE FINAL UI
# PAGE FLOW HELPERS FOR PATIENT + DOCTOR SEPARATE PORTALS
# ============================================================

import pandas as pd

def logout_to_home():
    return (
        gr.update(visible=True),   # home_page
        gr.update(visible=False),  # patient_register_page
        gr.update(visible=False),  # patient_login_page
        gr.update(visible=False),  # patient_link_page
        gr.update(visible=False),  # patient_portal_page
        gr.update(visible=False),  # doctor_register_page
        gr.update(visible=False),  # doctor_login_page
        gr.update(visible=False),  # doctor_dashboard_page
    )

# -------------------------
# HOME NAVIGATION
# -------------------------
def go_patient_entry():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=True),   # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def go_doctor_entry():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=True),   # doctor dashboard? no
    )

# Better explicit doctor flow:
def go_doctor_login():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=True),   # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def go_patient_register():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=True),   # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def back_to_patient_login():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=True),   # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def go_doctor_register():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=True),   # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def back_to_doctor_login():
    return (
        gr.update(visible=False),  # home
        gr.update(visible=False),  # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=True),   # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

# -------------------------
# REGISTRATION WRAPPERS
# -------------------------
def patient_register_and_message(name, username, email, password):
    msg = register_user("patient", name, username, email, password)
    return msg

def doctor_register_and_message(name, username, email, password):
    msg = register_user("doctor", name, username, email, password)
    return msg

# -------------------------
# LOGIN WRAPPERS WITH PAGE TRANSITIONS
# -------------------------
def patient_login_flow(login_id, password):
    ok, msg = login_user(login_id, password, "patient")

    if ok:
        return (
            msg,                     # status
            login_id,                # hidden patient session
            gr.update(visible=False),# home
            gr.update(visible=False),# patient register
            gr.update(visible=False),# patient login
            gr.update(visible=True), # patient link
            gr.update(visible=False),# patient portal
            gr.update(visible=False),# doctor register
            gr.update(visible=False),# doctor login
            gr.update(visible=False) # doctor dashboard
        )
    else:
        return (
            msg,
            "",
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update()
        )

def doctor_login_flow(login_id, password):
    ok, msg = login_user(login_id, password, "doctor")

    if ok:
        return (
            msg,                     # status
            login_id,                # hidden doctor session
            gr.update(visible=False),# home
            gr.update(visible=False),# patient register
            gr.update(visible=False),# patient login
            gr.update(visible=False),# patient link
            gr.update(visible=False),# patient portal
            gr.update(visible=False),# doctor register
            gr.update(visible=False),# doctor login
            gr.update(visible=True)  # doctor dashboard
        )
    else:
        return (
            msg,
            "",
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update()
        )

# -------------------------
# PATIENT LINK FLOW
# -------------------------
def patient_link_flow(doctor_id, patient_id):
    msg = link_patient_to_doctor(doctor_id, patient_id)

    success = "success" in str(msg).lower()

    if success:
        return (
            msg,
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=False),  # patient link
            gr.update(visible=True),   # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=False),  # doctor dashboard
        )
    else:
        return (
            msg,
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update(),
            gr.update()
        )

# -------------------------
# PATIENT ANALYZE WRAPPER
# -------------------------
def patient_analyze_with_gauge(mode, patient_email, audio_file, text_input):
    score, risk, mode_used, a_score, t_score, used_text, alert = patient_analyze(
        mode, patient_email, audio_file, text_input
    )

    gauge = None
    try:
        if score not in ["", None]:
            gauge = risk_gauge(float(score))
    except:
        gauge = None

    return score, risk, alert, a_score, t_score, mode_used, used_text, gauge

# -------------------------
# DOCTOR SUMMARY WRAPPER
# -------------------------
def doctor_view_patient_summary_only(selected_patient):
    if not selected_patient:
        empty_df = pd.DataFrame({"Message": ["No patient selected"]})
        return (
            "N/A", "No data", "No data", "No history",
            empty_df, "No alerts"
        )

    patient_email = selected_patient.split("|")[-1].strip()
    summary = get_latest_patient_summary(patient_email)
    df = get_patient_assessments(patient_email)

    alert_msg = "No alerts"

    if not df.empty:
        latest_alert = df.iloc[-1]["alert_level"]
        if latest_alert == "CRITICAL":
            alert_msg = "🚨 CRITICAL: Immediate doctor attention required."
        elif latest_alert == "HIGH":
            alert_msg = "⚠️ HIGH: Patient condition needs close monitoring."
        elif latest_alert == "MODERATE":
            alert_msg = "🟡 MODERATE: Continue follow-up."

    return (
        summary["latest_score"],
        summary["risk_label"],
        summary["alert_level"],
        summary["trend"],
        df if not df.empty else pd.DataFrame({"Message": ["No assessments found"]}),
        alert_msg
    )

# -------------------------
# DOCTOR PLOT HELPERS (ONE PLOT AT A TIME)
# -------------------------
def show_patient_risk_gauge_only(selected_patient):
    if not selected_patient:
        return None, None, "No patient selected."

    patient_email = selected_patient.split("|")[-1].strip()
    summary = get_latest_patient_summary(patient_email)

    if summary["latest_score"] == "N/A":
        return None, None, "No assessment data available."

    fig = risk_gauge(float(summary["latest_score"]))
    return fig, None, f"Showing risk gauge for {patient_email}"

def show_patient_memory_tracking_only(selected_patient):
    if not selected_patient:
        return None, None, "No patient selected."

    patient_email = selected_patient.split("|")[-1].strip()
    df = get_patient_assessments(patient_email)

    if df.empty:
        return None, None, "No assessment history available."

    fig = memory_decline_chart(patient_email)
    return None, fig, f"Showing memory decline tracking for {patient_email}"

In [ ]:
# ============================================================
# ADD BELOW doctor_view_patient(...)
# BUTTON-BASED DOCTOR VISUALS
# ============================================================

def show_patient_risk_gauge(selected_patient):
    if not selected_patient:
        return None, "No patient selected."

    patient_email = selected_patient.split("|")[-1].strip()
    summary = get_latest_patient_summary(patient_email)

    if summary["latest_score"] == "N/A":
        return None, "No assessment data available for this patient."

    fig = risk_gauge(float(summary["latest_score"]))
    msg = f"Showing current risk gauge for {patient_email}"
    return fig, msg


def show_patient_memory_tracking(selected_patient):
    if not selected_patient:
        return None, "No patient selected."

    patient_email = selected_patient.split("|")[-1].strip()

    df = get_patient_assessments(patient_email)
    if df.empty:
        return None, "No assessment history available for this patient."

    fig = memory_decline_chart(patient_email)
    msg = f"Showing memory decline tracking for {patient_email}"
    return fig, msg

In [ ]:
# ============================================================
# HELPER FUNCTIONS FOR CORRECTED REGISTER / LOGIN FLOW
# ADD THIS BEFORE UI CELL
# ============================================================

import gradio as gr

# ------------------------------------------------------------
# REGISTER HELPERS
# ------------------------------------------------------------
def patient_register_and_message(name, username, email, password):
    if not name or not username or not email or not password:
        return "❌ Please fill all patient registration fields."
    return register_user("patient", name, username, email, password)

def doctor_register_and_message(name, username, email, password):
    if not name or not username or not email or not password:
        return "❌ Please fill all doctor registration fields."
    return register_user("doctor", name, username, email, password)

# ------------------------------------------------------------
# PAGE VISIBILITY HELPERS
# Output order everywhere:
# [home, patient_register, patient_login, patient_link, patient_portal,
#  doctor_register, doctor_login, doctor_dashboard]
# ------------------------------------------------------------
def show_home():
    return (
        gr.update(visible=True),   # home
        gr.update(visible=False),  # patient register
        gr.update(visible=False),  # patient login
        gr.update(visible=False),  # patient link
        gr.update(visible=False),  # patient portal
        gr.update(visible=False),  # doctor register
        gr.update(visible=False),  # doctor login
        gr.update(visible=False),  # doctor dashboard
    )

def go_home_to_patient():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),   # patient login
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    )

def go_home_to_doctor():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),   # doctor login
        gr.update(visible=False),
    )

def go_patient_register():
    return (
        gr.update(visible=False),
        gr.update(visible=True),   # patient register
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    )

def back_to_patient_login():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),   # patient login
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    )

def go_doctor_register():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),   # doctor register
        gr.update(visible=False),
        gr.update(visible=False),
    )

def back_to_doctor_login():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),   # doctor login
        gr.update(visible=False),
    )

def logout_to_home():
    return show_home()

# ------------------------------------------------------------
# LOGIN FLOWS
# ------------------------------------------------------------
def patient_login_flow(login_id, password):
    ok, msg = login_user(login_id, password, "patient")

    if ok:
        return (
            f"✅ {msg}",
            login_id,  # patient session
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=True),   # patient link
            gr.update(visible=False),  # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=False),  # doctor dashboard
        )
    else:
        return (
            f"❌ {msg}",
            "",
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=True),   # stay on patient login
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        )

def doctor_login_flow(login_id, password):
    ok, msg = login_user(login_id, password, "doctor")

    if ok:
        return (
            f"✅ {msg}",
            login_id,  # doctor session
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=False),  # patient link
            gr.update(visible=False),  # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=True),   # doctor dashboard
        )
    else:
        return (
            f"❌ {msg}",
            "",
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),   # stay on doctor login
            gr.update(visible=False),
        )

# ------------------------------------------------------------
# FLEXIBLE LINK FLOW (IMPORTANT FIX)
# If already linked -> STILL GO TO PATIENT PORTAL
# ------------------------------------------------------------
def patient_link_flow(doctor_id, patient_id):
    if not doctor_id or not patient_id:
        return (
            "❌ Please enter both doctor and patient email/username.",
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=True),   # stay on patient link
            gr.update(visible=False),  # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=False),  # doctor dashboard
        )

    # Reuse your DB function if you already have one
    # Must return string message
    msg = link_patient_to_doctor(doctor_id, patient_id)

    # IMPORTANT:
    # Even if already linked -> allow to continue
    if ("already linked" in msg.lower()) or ("linked successfully" in msg.lower()) or ("success" in msg.lower()):
        return (
            f"✅ {msg}",
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=False),  # patient link
            gr.update(visible=True),   # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=False),  # doctor dashboard
        )
    else:
        return (
            f"❌ {msg}",
            gr.update(visible=False),  # home
            gr.update(visible=False),  # patient register
            gr.update(visible=False),  # patient login
            gr.update(visible=True),   # stay on link page
            gr.update(visible=False),  # patient portal
            gr.update(visible=False),  # doctor register
            gr.update(visible=False),  # doctor login
            gr.update(visible=False),  # doctor dashboard
        )

In [ ]:
# ============================================================
# COOKIE THEFT IMAGE PATH (CORRECT FOR YOUR DRIVE ROOT)
# ADD THIS BEFORE FINAL UI CELL
# ============================================================

import os

COOKIE_IMG_PATH = "/content/drive/MyDrive/cookie_theft.png"

if not os.path.exists(COOKIE_IMG_PATH):
    COOKIE_IMG_PATH = None
    print("⚠️ Cookie Theft image not found.")
else:
    print("✅ Cookie Theft image found:", COOKIE_IMG_PATH)

✅ Cookie Theft image found: /content/drive/MyDrive/cookie_theft.png


In [ ]:
# ============================================================
# FINAL CORRECTED PAGE-BASED UI
# PATIENT + DOCTOR SEPARATE
# NO TABS
# ============================================================

import gradio as gr
import pandas as pd

CUSTOM_CSS = """
body {
    background: #f3f3f5 !important;
}
.gradio-container {
    background: #f3f3f5 !important;
    font-family: 'Segoe UI', sans-serif !important;
    color: #2f2f3a !important;
    max-width: 1600px !important;   /* increased width */
    margin: auto !important;
    padding-left: 20px !important;
    padding-right: 20px !important;
}
.block, .card {
    background: rgba(255,255,255,0.95) !important;
    border-radius: 10px !important;
    border: 1px solid #e4e4ea !important;
    box-shadow: 0 4px 14px rgba(0,0,0,0.05) !important;
    padding: 14px !important;
}
h1, h2, h3, h4 {
    color: #2f2f3a !important;
    font-weight: 700 !important;
}
label {
    color: #6b5ca5 !important;
    font-weight: 600 !important;
}
input, textarea, select {
    background: #ffffff !important;
    border: 1px solid #d9d9e3 !important;
    border-radius: 8px !important;
    color: #2f2f3a !important;
    box-shadow: none !important;
}
button {
    border-radius: 8px !important;
    font-weight: 600 !important;
}
button.primary, .gr-button-primary {
    background: #d9ccff !important;
    color: #2f2f3a !important;
    border: 1px solid #c7b6ff !important;
}
button.secondary {
    background: #efe9ff !important;
    color: #5c4c99 !important;
    border: 1px solid #d9cdff !important;
}
.hero-box {
    background: linear-gradient(135deg, rgba(255,255,255,0.96), rgba(248,245,255,0.96)) !important;
    border: 1px solid #e4ddff !important;
    border-radius: 16px !important;
    padding: 28px !important;
    text-align: center !important;
}
.page-banner {
    border-radius: 14px !important;
    padding: 16px !important;
    margin-bottom: 16px !important;
    background: linear-gradient(135deg, rgba(255,255,255,0.94), rgba(242,237,255,0.94)) !important;
    border: 1px solid #e5ddff !important;
}
.small-top-btn {
    max-width: 110px !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet"),
    css=CUSTOM_CSS,
    title="MemoryCare Dashboard"
) as app:

    # --------------------------------------------------------
    # SESSION STATE
    # --------------------------------------------------------
    patient_session = gr.State("")
    doctor_session = gr.State("")

    # ========================================================
    # HOME PAGE
    # ========================================================
    home_page = gr.Column(visible=True)
    with home_page:
        gr.Markdown("""
        <div class='hero-box'>
            <h1>🧠 MemoryCare Dashboard</h1>
            <h3>Early Cognitive Risk Screening System</h3>
            <p><b>Separate Patient Portal + Separate Doctor Dashboard</b></p>
            <p>Choose your portal to continue.</p>
        </div>
        """)

        with gr.Row():
            patient_portal_btn = gr.Button("Patient Portal", variant="primary")
            doctor_portal_btn = gr.Button("Doctor Portal", variant="secondary")

    # ========================================================
    # PATIENT REGISTER PAGE
    # ========================================================
    patient_register_page = gr.Column(visible=False)
    with patient_register_page:
        with gr.Row():
            gr.Markdown("## 👤 Patient Registration")
            patient_reg_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>New Patient Registration</h3>
            <p>If you are a new patient, register first and then go back to login.</p>
        </div>
        """)

        p_reg_name = gr.Textbox(label="Full Name")
        p_reg_username = gr.Textbox(label="Username")
        p_reg_email = gr.Textbox(label="Email")
        p_reg_password = gr.Textbox(label="Password", type="password")

        with gr.Row():
            p_register_btn = gr.Button("Register as Patient", variant="primary")
            p_back_to_login_btn = gr.Button("Back to Patient Login", variant="secondary")

        p_reg_status = gr.Textbox(label="Registration Status")

        p_register_btn.click(
            fn=patient_register_and_message,
            inputs=[p_reg_name, p_reg_username, p_reg_email, p_reg_password],
            outputs=p_reg_status
        )

    # ========================================================
    # PATIENT LOGIN PAGE
    # ========================================================
    patient_login_page = gr.Column(visible=False)
    with patient_login_page:
        with gr.Row():
            gr.Markdown("## 🔐 Patient Login")
            patient_login_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Patient Login</h3>
            <p>Already registered? Login below. New patient? Click register first.</p>
        </div>
        """)

        p_new_patient_btn = gr.Button("New Patient? Register", variant="secondary")

        p_login_id = gr.Textbox(label="Patient Email or Username")
        p_login_password = gr.Textbox(label="Password", type="password")

        p_login_btn = gr.Button("Patient Login", variant="primary")
        p_login_status = gr.Textbox(label="Login Status")

    # ========================================================
    # PATIENT LINK PAGE
    # ========================================================
    patient_link_page = gr.Column(visible=False)
    with patient_link_page:
        with gr.Row():
            gr.Markdown("## 🔗 Link to Your Doctor")
            patient_link_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Link to Doctor</h3>
            <p>Enter your doctor email/username and your patient email/username.</p>
        </div>
        """)

        p_link_doctor = gr.Textbox(label="Doctor Email or Username")
        p_link_patient = gr.Textbox(label="Patient Email or Username")

        p_link_btn = gr.Button("Link to Doctor", variant="primary")
        p_link_status = gr.Textbox(label="Link Status")

    # ========================================================
    # PATIENT PORTAL PAGE
    # ========================================================
    patient_portal_page = gr.Column(visible=False)
    with patient_portal_page:
        with gr.Row():
            gr.Markdown("## 🧠 Patient Assessment Portal")
            patient_portal_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Cookie Theft Assessment</h3>
            <p>Please describe what you see in the image below using audio, text, or both.</p>
        </div>
        """)

        cookie_img = gr.Image(
            value=COOKIE_IMG_PATH if 'COOKIE_IMG_PATH' in globals() else None,
            label="Cookie Theft Picture",
            interactive=False,
            height=350
        )

        patient_email_input = gr.Textbox(label="Patient Email")

        mode_input = gr.Dropdown(
            ["Audio Only", "Text Only", "Audio + Text"],
            value="Audio Only",
            label="Select Analysis Mode"
        )

        with gr.Row():
            audio_input = gr.Audio(type="filepath", label="Record / Upload Speech")
            text_input = gr.Textbox(
                label="Typed Description / Patient Text",
                lines=8,
                placeholder="Describe the Cookie Theft picture here..."
            )

        analyze_btn = gr.Button("Analyze Memory Risk", variant="primary")

        with gr.Row():
            final_score_output = gr.Textbox(label="Final Risk Score")
            risk_label_output = gr.Textbox(label="Risk Interpretation")
            alert_output = gr.Textbox(label="Alert Level")

        with gr.Row():
            audio_score_output = gr.Textbox(label="Audio Score")
            text_score_output = gr.Textbox(label="Text Score")

        mode_output = gr.Textbox(label="Input Mode Used")
        used_text_output = gr.Textbox(label="Transcript / Used Text", lines=6)

        patient_gauge = gr.Plot(label="Current Risk Gauge")

        def patient_analyze_with_gauge(mode, patient_email, audio_file, text_input):
            score, risk, mode_used, a_score, t_score, used_text, alert = patient_analyze(
                mode, patient_email, audio_file, text_input
            )

            try:
                gauge = risk_gauge(float(score))
            except:
                gauge = None

            return score, risk, alert, a_score, t_score, mode_used, used_text, gauge

        analyze_btn.click(
            fn=patient_analyze_with_gauge,
            inputs=[mode_input, patient_email_input, audio_input, text_input],
            outputs=[
                final_score_output,
                risk_label_output,
                alert_output,
                audio_score_output,
                text_score_output,
                mode_output,
                used_text_output,
                patient_gauge
            ]
        )

    # ========================================================
    # DOCTOR REGISTER PAGE
    # ========================================================
    doctor_register_page = gr.Column(visible=False)
    with doctor_register_page:
        with gr.Row():
            gr.Markdown("## 🩺 Doctor Registration")
            doctor_reg_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Doctor Registration</h3>
            <p>If you are a new doctor, register first and then return to login.</p>
        </div>
        """)

        d_reg_name = gr.Textbox(label="Full Name")
        d_reg_username = gr.Textbox(label="Username")
        d_reg_email = gr.Textbox(label="Email")
        d_reg_password = gr.Textbox(label="Password", type="password")

        with gr.Row():
            d_register_btn = gr.Button("Register as Doctor", variant="primary")
            d_back_to_login_btn = gr.Button("Back to Doctor Login", variant="secondary")

        d_reg_status = gr.Textbox(label="Registration Status")

        d_register_btn.click(
            fn=doctor_register_and_message,
            inputs=[d_reg_name, d_reg_username, d_reg_email, d_reg_password],
            outputs=d_reg_status
        )

    # ========================================================
    # DOCTOR LOGIN PAGE
    # ========================================================
    doctor_login_page = gr.Column(visible=False)
    with doctor_login_page:
        with gr.Row():
            gr.Markdown("## 🔐 Doctor Login")
            doctor_login_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Doctor Login</h3>
            <p>Already registered? Login below. New doctor? Register first.</p>
        </div>
        """)

        d_new_doctor_btn = gr.Button("New Doctor? Register", variant="secondary")

        d_login_id = gr.Textbox(label="Doctor Email or Username")
        d_login_password = gr.Textbox(label="Password", type="password")

        d_login_btn = gr.Button("Doctor Login", variant="primary")
        d_login_status = gr.Textbox(label="Login Status")

    # ========================================================
    # DOCTOR DASHBOARD PAGE
    # ========================================================
    doctor_dashboard_page = gr.Column(visible=False)
    with doctor_dashboard_page:
        with gr.Row():
            gr.Markdown("## 🩺 Doctor Dashboard")
            doctor_dash_logout = gr.Button("Logout", variant="secondary")

        gr.Markdown("""
        <div class='page-banner'>
            <h3>Doctor Dashboard</h3>
            <p>Load patients, view summary, risk gauge, and memory decline tracking.</p>
        </div>
        """)

        doctor_email_dashboard = gr.Textbox(label="Doctor Email")
        load_patients_btn = gr.Button("Load My Patients", variant="primary")

        patient_selector = gr.Dropdown(label="Select Patient")
        patient_list_table = gr.Dataframe(label="My Patients")

        view_patient_btn = gr.Button("View Selected Patient Summary", variant="primary")

        with gr.Row():
            latest_score_box = gr.Textbox(label="Latest Risk Score")
            latest_risk_box = gr.Textbox(label="Risk Interpretation")
            latest_alert_box = gr.Textbox(label="Alert Level")
            trend_box = gr.Textbox(label="Trend")

        alerts_box = gr.Textbox(label="Doctor Alerts / Insights", lines=3)
        assessment_table = gr.Dataframe(label="Patient Assessment History")

        gr.Markdown("### Doctor Analysis Tools")

        with gr.Row():
            show_gauge_btn = gr.Button("Risk Gauge", variant="secondary")
            show_memory_btn = gr.Button("Memory Decline Tracking", variant="secondary")

        doctor_visual_status = gr.Textbox(label="Visualization Status")

        with gr.Row():
            doctor_gauge_plot = gr.Plot(label="Risk Gauge")
            memory_trend_plot = gr.Plot(label="Memory Decline Tracking")

    # ========================================================
    # OUTPUT TARGETS
    # ========================================================
    nav_targets = [
        home_page, patient_register_page, patient_login_page, patient_link_page,
        patient_portal_page, doctor_register_page, doctor_login_page, doctor_dashboard_page
    ]

    # ========================================================
    # NAVIGATION
    # ========================================================
    patient_portal_btn.click(fn=go_home_to_patient, outputs=nav_targets)
    doctor_portal_btn.click(fn=go_home_to_doctor, outputs=nav_targets)

    p_new_patient_btn.click(fn=go_patient_register, outputs=nav_targets)
    p_back_to_login_btn.click(fn=back_to_patient_login, outputs=nav_targets)

    d_new_doctor_btn.click(fn=go_doctor_register, outputs=nav_targets)
    d_back_to_login_btn.click(fn=back_to_doctor_login, outputs=nav_targets)

    # ========================================================
    # LOGIN FLOWS
    # ========================================================
    p_login_btn.click(
        fn=patient_login_flow,
        inputs=[p_login_id, p_login_password],
        outputs=[p_login_status, patient_session] + nav_targets
    )

    d_login_btn.click(
        fn=doctor_login_flow,
        inputs=[d_login_id, d_login_password],
        outputs=[d_login_status, doctor_session] + nav_targets
    )

    # ========================================================
    # LINK FLOW
    # ========================================================
    p_link_btn.click(
        fn=patient_link_flow,
        inputs=[p_link_doctor, p_link_patient],
        outputs=[p_link_status] + nav_targets
    )

    # ========================================================
    # DOCTOR ACTIONS
    # ========================================================
    load_patients_btn.click(
        fn=get_doctor_patients,
        inputs=doctor_email_dashboard,
        outputs=patient_list_table
    )

    load_patients_btn.click(
        fn=doctor_patient_dropdown_choices,
        inputs=doctor_email_dashboard,
        outputs=patient_selector
    )

    view_patient_btn.click(
        fn=doctor_view_patient_summary_only,
        inputs=patient_selector,
        outputs=[
            latest_score_box,
            latest_risk_box,
            latest_alert_box,
            trend_box,
            assessment_table,
            alerts_box
        ]
    )

    show_gauge_btn.click(
        fn=show_patient_risk_gauge_only,
        inputs=patient_selector,
        outputs=[doctor_gauge_plot, memory_trend_plot, doctor_visual_status]
    )

    show_memory_btn.click(
        fn=show_patient_memory_tracking_only,
        inputs=patient_selector,
        outputs=[doctor_gauge_plot, memory_trend_plot, doctor_visual_status]
    )

    # ========================================================
    # LOGOUTS
    # ========================================================
    patient_reg_logout.click(fn=logout_to_home, outputs=nav_targets)
    patient_login_logout.click(fn=logout_to_home, outputs=nav_targets)
    patient_link_logout.click(fn=logout_to_home, outputs=nav_targets)
    patient_portal_logout.click(fn=logout_to_home, outputs=nav_targets)

    doctor_reg_logout.click(fn=logout_to_home, outputs=nav_targets)
    doctor_login_logout.click(fn=logout_to_home, outputs=nav_targets)
    doctor_dash_logout.click(fn=logout_to_home, outputs=nav_targets)

# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=False,
    debug=False,
    inline=True,
    prevent_thread_lock=True
)

HOME_BG_PATH exists: True /content/drive/MyDrive/home.jpg
COOKIE_IMG_PATH exists: True /content/drive/MyDrive/cookie_theft.png
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
